# Lab Sheet 7: Building a PoS Tagger

Aim: To design and implement a custom PoS tagger using rule-based, statistical (Hidden Markov Model) and/or machine learning(CRF/BiLSTM) and evaluate its performance using accuracy and confusion matrix.  

1. Rule Based Tagging
2. Statistical Tagging (HMM)
3. Machine Learning Approaches 

In [ ]:
import nltk
from nltk.corpus import treebank

In [ ]:
sentences = treebank.tagged_sents()

train_data = sentences[:3000]
test_data = sentences[3000:]

In [ ]:
#Rule-Based PoS Tagger
from nltk.tag import RegexpTagger, DefaultTagger

patterns = [
    (r'.*ing$', 'VBG'),
    (r'.*ed$', 'VBD'),
    (r'.*s$', 'NNS'),
    (r'.*', 'NN') # default
]

rule_tagger = RegexpTagger(patterns)
print(rule_tagger.tag(["playing", "dogs", "walked"]))

default_tagger = DefaultTagger('NN') # assign noun to all
accuracy = default_tagger.accuracy(test_data)

print("Rule-Based Accuracy:", accuracy)

In [ ]:
# Statistical PoS Tagger (HMM)
from nltk.tag import hmm

trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train(train_data)

# Test sentence
print(hmm_tagger.tag(["The", "dog", "runs"]))

# Accuracy
accuracy = hmm_tagger.accuracy(test_data)
print("HMM Accuracy:", accuracy)

In [ ]:
# Machine Learning PoS Tagger
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def word_features(word):
    return {
        'word': word,
        'is_upper': word.isupper(),
        'is_digit': word.isdigit(),
        'suffix': word[-2:]
    }

X = []
y = []

for sent in train_data:
    for word, tag in sent:
        X.append(word_features(word))
        y.append(tag)
    
vec = DictVectorizer()
X_vec = vec.fit_transform(X)

model = LogisticRegression(max_iter=200)
model.fit(X_vec, y)

In [ ]:
# Test on one sentence

test_words = ["The", "cat", "sat"]
test_features = [word_features(w) for w in test_words]

test_vec = vec.transform(test_features)

print(list(zip(test_words, model.predict(test_vec))))